In [1]:
import uproot
import dask_awkward as dak
import awkward as ak
import numpy as np
import pandas as pd
import vector
from dask.diagnostics import ProgressBar

In [2]:
vector.register_awkward()

In [3]:
tree_path = "Events"
base_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoKstar2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoKstarMuMu_NoFilter_2022/260427_132612/0000"
mc_files = {f"{base_path}/*.root": tree_path}

In [4]:
def get_b0_truth_mask(df):
    # --- Constantes PDG ---
    PDG_B0        = 511
    PDG_KSTAR     = 313
    PDG_KAON_plus = 321
    PDG_PION_neg  = -211
    PDG_MUON_plus = -13
    PDG_MUON_neg  = 13
    PDG_PHOTON    = 22

    mu1_idx  = df["BPHMuon_genPartIdx"][df["BToTrkTrkMuMu_l1_idx"]]
    mu2_idx  = df["BPHMuon_genPartIdx"][df["BToTrkTrkMuMu_l2_idx"]]
    trk1_idx = df["Track_genPartIdx"][df["BToTrkTrkMuMu_trk1_idx"]]
    trk2_idx = df["Track_genPartIdx"][df["BToTrkTrkMuMu_trk2_idx"]]

    # Verificação de Identidade (PDG ID)
    mu1_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mu2_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu2_idx, mu2_idx >= 0)]
    trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(trk1_idx, trk1_idx >= 0)]
    trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(trk2_idx, trk2_idx >= 0)]

    flavor_match = (trk1_pdg == PDG_KAON_plus) & (trk2_pdg == PDG_PION_neg) & \
                   (mu1_pdg == PDG_MUON_plus) & (mu2_pdg == PDG_MUON_neg)

    # Linhagem do Hadron (K, Pi -> K* -> B0)
    mom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk1_idx, trk1_idx >= 0)]
    mom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk2_idx, trk2_idx >= 0)]
    
    mom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    mom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    # K e Pi devem vir do mesmo objeto K*0
    match_kstar = (mom_trk1_idx == mom_trk2_idx) & (mom_trk1_idx >= 0) & \
                  (mom_trk1_pdg == PDG_KSTAR) & (mom_trk2_pdg == PDG_KSTAR)
    
    # O K*0 deve vir de um B0
    gmom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    gmom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    gmom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk1_idx, gmom_trk1_idx >= 0)]
    gmom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk2_idx, gmom_trk2_idx >= 0)]
    
    match_kstar_to_b0 = (gmom_trk1_idx == gmom_trk2_idx) & (gmom_trk1_idx >= 0) & \
                        (gmom_trk1_pdg == PDG_B0) & (gmom_trk2_pdg == PDG_B0)

    mom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu2_idx, mu2_idx >= 0)]
    mom_mu1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    mom_mu2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]
    
    gmom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    gmom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]

    # Muon 1: Mae e o B0 do hadron OU (Mae e foton e Avo e o B0 do hadron)
    mu1_from_same_b0 = (mom_mu1_idx == gmom_trk1_idx) | \
                       ((mom_mu1_pdg == PDG_PHOTON) & (gmom_mu1_idx == gmom_trk1_idx))

    mu2_from_same_b0 = (mom_mu2_idx == gmom_trk1_idx) | \
                       ((mom_mu2_pdg == PDG_PHOTON) & (gmom_mu2_idx == gmom_trk1_idx))

    # --- Mascara Final de Truth Matching ---
    full_mask = flavor_match & match_kstar & match_kstar_to_b0 & \
                mu1_from_same_b0 & mu2_from_same_b0
    
    return dak.fill_none(full_mask, False)

In [5]:
def apply_selection(df, mode="mc"):

    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    mask = (
        #(((df['MuMu_mass'][df["BToTrkTrkMuMu_ll_idx"]] > 1.0) & (df['MuMu_mass'][df["BToTrkTrkMuMu_ll_idx"]] < 2.7)) | 
        #((df['MuMu_mass'][df["BToTrkTrkMuMu_ll_idx"]] > 4.0) & (df['MuMu_mass'][df["BToTrkTrkMuMu_ll_idx"]] < 6.0))) & 
        #(mask_kpi_window | mask_pik_window) &         
        #(is_kpi_closer) &
        
        (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l1_idx"]] > 2.0) & (df['BPHMuon_pt'][df["BToTrkTrkMuMu_l2_idx"]] > 2.0) & 
        (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l1_idx"]]) < 2.4) & (abs(df['BPHMuon_eta'][df["BToTrkTrkMuMu_l2_idx"]]) < 2.4) &
        (df['BToTrkTrkMuMu_fit_trk1_pt'] > 1.5) &
        (df['BToTrkTrkMuMu_fit_trk2_pt'] > 1.5) 
    )

    cols_to_keep = [
            'BToTrkTrkMuMu_fit_mass_Kpi',
            'BToTrkTrkMuMu_mll_fullfit',
        ]
    
        
    return df[cols_to_keep][mask]

In [6]:
def build_dataframe(file_dict, label="Dataset", mode="mc"):
    print(f"\nProcessando {label}...")

    df = uproot.dask(file_dict)
    df = apply_selection(df, mode=mode)

    with ProgressBar():
        awkward_array = df.compute()
        df_pandas = ak.to_dataframe(awkward_array).reset_index(drop=True)

    initial_len = len(df_pandas)
    df_pandas = df_pandas.replace([np.inf, -np.inf], np.nan)
    df_pandas = df_pandas.dropna()
    final_len  = len(df_pandas)
    removed    = initial_len - final_len

    if removed > 0:
        print(f"  [Limpeza] Removidos {removed} eventos com NaN ou Inf ({removed/initial_len:.2%})")

    print(f"{label} finalizado. Linhas: {final_len}")
    return df_pandas

In [7]:
df_mc   = build_dataframe(mc_files,   "MC Signal (Truth Matched)", mode="mc")


Processando MC Signal (Truth Matched)...
[########################################] | 100% Completed | 102.42 ms
[########################################] | 100% Completed | 10.56 s
MC Signal (Truth Matched) finalizado. Linhas: 17727


In [8]:
df_mc['q2'] = df_mc['BToTrkTrkMuMu_mll_fullfit']**2

bins = [1.1, 2, 4.3, 6, 8.68, 10.09, 12.86, 14.18, 16, 19]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df_mc['bin_index'] = pd.cut(df_mc['q2'], bins=bins, labels=labels, right=False)
df_mc['bin_index'] = pd.cut(df_mc['q2'], bins=bins, labels=labels, right=False).astype(float)

In [9]:
print(f"Total de partículas no DF RECO: {len(df_mc)}")

for i in range(9):  
    is_bin = df_mc["bin_index"] == i    
    df_filtrado = df_mc[is_bin]    
    print(f"Bin {i}: {len(df_filtrado)} partículas B0 encontradas")

Total de partículas no DF RECO: 17727
Bin 0: 440 partículas B0 encontradas
Bin 1: 1182 partículas B0 encontradas
Bin 2: 1014 partículas B0 encontradas
Bin 3: 2198 partículas B0 encontradas
Bin 4: 1496 partículas B0 encontradas
Bin 5: 3606 partículas B0 encontradas
Bin 6: 1860 partículas B0 encontradas
Bin 7: 2413 partículas B0 encontradas
Bin 8: 2140 partículas B0 encontradas
